# Exploring Hugging Face Pipelines

My notes and experiments with Hugging Face's `pipelines` API — the high-level
interface for running inference with pretrained open-source models, without
having to deal with tokenizers, model classes, or any of the lower-level
plumbing directly.

The core idea is just two steps:

```python
my_pipeline = pipeline("the_task_I_want_to_do")
result = my_pipeline(my_input)
```

That's it. In this notebook I try out pipelines across several task types —
sentiment analysis, NER, question answering, summarization, translation,
zero-shot classification, text generation, image generation, and text-to-speech —
to get a feel for how consistent and easy the API is across very different model
types.

**Environment:** built to run on Google Colab with a T4 GPU (`Runtime > Change
runtime type > T4 GPU`), since several of these models are heavy enough to need
GPU acceleration to run in reasonable time.


## Quick note: training vs. inference

Before diving in, worth being clear on a distinction that comes up constantly
when working with these models:

**Training** is when a model is given data to learn from, updating its internal
parameters (weights) so it gets better at a task. Training a model that's
already been trained further is called *fine-tuning*.

**Inference** is using a model that's *already trained*, to produce outputs on
new inputs — taking advantage of everything it learned during training.
Sometimes called "execution" or "running a model."

Every time I've called the GPT/Claude/Gemini APIs, that's inference — the "P" in
GPT literally stands for "Pre-trained." The `pipelines` API here is also purely
for inference: it's meant for running models that already exist, not training
new ones.


## A couple of things I want to remember about Colab

- Data science code throws a lot of warnings/messages that are mostly safe to
  ignore — skim them, and if something breaks later they might hint at why.
- If I ever see an error like:

  > `Runtime error: CUDA is required but not available for bitsandbytes...`

  this is misleading — it's *not* actually a package version problem. It
  usually means Colab swapped out the runtime underneath me (often because Colab
  was busy). Fix:
  1. `Runtime` menu → Disconnect and delete runtime
  2. Reload the notebook fresh, `Edit` menu → Clear All Outputs
  3. Reconnect to a new T4 (top-right button)
  4. Check "View resources" to confirm the GPU is actually attached
  5. Re-run all cells from the top, starting with the pip installs


## Setup

Installing the libraries first (pip installs should always be at the top —
if the runtime ever resets, these need to be re-run).


In [ ]:
# Pip installs should come at the top.
# If the runtime ever resets, this needs to be run again.

!pip install -q --upgrade datasets==3.6.0


In [ ]:
!pip install "transformers<5"


### Confirming the GPU

Checking that Colab actually gave me a T4 GPU before running anything heavy.


In [ ]:
# Should be a Tesla T4

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)
  if gpu_info.find('Tesla T4') >= 0:
    print("Success - Connected to a T4")
  else:
    print("NOT CONNECTED TO A T4")


In [ ]:
# Imports

import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import pipeline
from diffusers import DiffusionPipeline
from datasets import load_dataset
import soundfile as sf
from IPython.display import Audio


### Hugging Face login

Need a free Hugging Face account (https://huggingface.co) and an API token with
**write** permissions, saved in Colab's Secrets panel under the key
`HF_TOKEN`, with notebook access switched on.


In [ ]:
hf_token = userdata.get('HF_TOKEN')
if hf_token and hf_token.startswith("hf_"):
  print("HF key looks good so far")
else:
  print("HF key is not set - please click the key in the left sidebar")
login(hf_token, add_to_git_credential=True)


## Using pipelines

The whole point of `pipeline()` is that it hides the plumbing (tokenization,
model loading, pre/post-processing) behind sensible defaults, so calling it
looks the same no matter what task I'm doing underneath.

**Step 1 — create a pipeline** (a callable object):

```python
my_pipeline = pipeline(task, model=xx, device=xx)
```

If I don't specify a model, Hugging Face picks a sensible default for the task.
`device="cuda"` uses the GPU (or `"mps"` on a Mac).

**Step 2 — call it** as many times as I want:

```python
my_pipeline(input1)
my_pipeline(input2)
```

Let's try this across a bunch of different tasks.


### Sentiment analysis

Starting with the default sentiment model.


In [ ]:
my_simple_sentiment_analyzer = pipeline("sentiment-analysis", device="cuda")
result = my_simple_sentiment_analyzer("I'm super excited to be on the way to LLM mastery!")
print(result)


In [ ]:
result = my_simple_sentiment_analyzer("I should be more excited to be on the way to LLM mastery!")
print(result)


The default model only outputs POSITIVE/NEGATIVE, which feels a bit coarse for
that second sentence — it reads more mixed than purely negative to me. Let's try
a model trained on a finer-grained 1–5 star rating scale instead, and compare.


In [ ]:
better_sentiment = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment", device="cuda")
result = better_sentiment("I am excited to be on the way to LLM mastery!!")
result_2 = better_sentiment('I should be more excited to be on the way to LLM mastery')
print(result)
print(result_2)


### Named Entity Recognition (NER)

Pulling out named entities (people, orgs, locations, etc.) from a sentence.


In [ ]:
ner = pipeline("ner", device="cuda")
result = ner("AI Engineers are learning about the amazing pipelines from HuggingFace in Google Colab")
for entity in result:
  print(entity)


### Question answering with context

Given a passage of context and a question, the model extracts the answer span
directly from the context.


In [ ]:
question = "What are Hugging Face pipelines?"
context = "Pipelines are a high level API for inference of LLMs"

question_answerer = pipeline("question-answering", device="cuda")
result = question_answerer(question=question, context=context)
print(result)


### Text summarization


In [ ]:
summarizer = pipeline("summarization", device="cuda")
text = """
The Hugging Face transformers library is an incredibly versatile and powerful tool for natural language processing (NLP).
It allows users to perform a wide range of tasks such as text classification, named entity recognition, and question answering, among others.
It's an extremely popular library that's widely used by the open-source data science community.
It lowers the barrier to entry into the field by providing Data Scientists with a productive, convenient way to work with transformer models.
"""

summary = summarizer(text, max_length=50, min_length=25, do_sample=False)
print(summary[0]['summary_text'])


### Translation

Trying a couple of translation pipelines, one with the default model and one
where I explicitly specify a model (browsable at
https://huggingface.co/models?pipeline_tag=translation&sort=trending).


In [ ]:
translator = pipeline("translation_en_to_fr", device="cuda")
result = translator("The Data Scientists were truly amazed by the power and simplicity of the HuggingFace pipeline API.")
print(result[0]['translation_text'])


In [ ]:
# Same idea, but explicitly specifying a model this time

translator = pipeline("translation_en_to_es", model="Helsinki-NLP/opus-mt-en-es", device="cuda")
result = translator("The Data Scientists were truly amazed by the power and simplicity of the HuggingFace pipeline API.")
print(result[0]['translation_text'])


### Zero-shot classification

No training needed for new labels — I just supply candidate categories at
inference time and the model scores how well the text fits each one.


In [ ]:
classifier = pipeline("zero-shot-classification", device="cuda")
result = classifier("Hugging Face's Transformers library is amazing!", candidate_labels=["AI", "sports", "politics"])
print(result)


### Text generation


In [ ]:
generator = pipeline("text-generation", device="cuda")
result = generator("If there's one thing I want you to remember about using HuggingFace pipelines, it's")
print(result[0]['generated_text'])


### Image generation

Pipelines aren't just for `transformers` — the `diffusers` library uses the
exact same pattern for diffusion models. Here I'm using SDXL-Turbo, a fast
distilled version of Stable Diffusion XL that only needs a handful of inference
steps.


In [ ]:
from IPython.display import display
from diffusers import AutoPipelineForText2Image
import torch

pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16")
pipe.to("cuda")
prompt = "A class of students learning AI engineering in a vibrant pop-art style"
image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
display(image)


### Audio generation (text-to-speech)

Text-to-speech using SpeechT5. This model needs a *speaker embedding* — a
vector that determines the voice characteristics — pulled from a dataset of
precomputed speaker embeddings (CMU ARCTIC).


In [ ]:
synthesiser = pipeline("text-to-speech", "microsoft/speecht5_tts", device='cuda')
embeddings_dataset = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)
speech = synthesiser("Hi to an artificial intelligence engineer, on the way to mastery!", forward_params={"speaker_embeddings": speaker_embedding})

Audio(speech["audio"], rate=speech["sampling_rate"])


## Reference: all available pipelines

For future reference — the full lists of pipeline tasks:

- **Transformers pipelines** (scroll down to the Tasks section, expand the
  parameters to see them all):
  https://huggingface.co/docs/transformers/main_classes/pipelines
- **Diffusers pipelines** (the diffusion-model equivalent, used above for image
  generation):
  https://huggingface.co/docs/diffusers/en/api/pipelines/overview

Worth coming back to this list whenever I want to try a new task type — the
API pattern (`pipeline(task) → call it`) stays the same no matter which one I
pick.
